# SineWave Example — GlyphViz Channel Animation Generator

This notebook generates four CSV files that demonstrate GlyphViz's channel
animation system with sinusoidal time-series data.

## Scene layout

| Column | Content |
|--------|---------|
| Root nodes (branch level 0) | One node per topology: Sphere, Torus, Pin, Rod, Point, Plane, Spiral — spread along the X axis |
| Child nodes (branch level 1) | 32 children per root, spread along the topology's natural `translate_x` dimension |
| Animation | `translate_z` of each child is driven by a sinusoidal track. Child *i* uses track *i* at frequency *i* Hz |

## Output files

- `SineWave_Example_gv_node.csv` — node hierarchy with `ch_input_id` assignments
- `SineWave_Example_gv_tag.csv` — topology name labels
- `SineWave_Example_gv_ch-map.csv` — channel-to-attribute mappings
- `SineWave_Example_gv_ch-tracks.csv` — 512 frames × 32 sinusoidal tracks

## Loading in GlyphViz

1. **File > Open Node CSV** → `SineWave_Example_gv_node.csv`
2. The **Channels** panel appears in the left sidebar automatically.
3. Press ▶ to animate.  Use the FPS spinner to adjust playback speed.

In [ ]:
import csv
import math
from pathlib import Path

## Parameters

Edit the values in this cell to customise the example.
Everything below is derived automatically.

In [ ]:
# Output location (same directory as this notebook by default)
OUTPUT_DIR = Path('.').resolve()
PREFIX     = 'SineWave_Example'

# Animation
N_CHILDREN            = 32    # child nodes per topology root
AMPLITUDE             = 3.0   # peak translate_z displacement (scene units)
MIN_SAMPLES_PER_CYCLE = 16    # min frames per cycle at highest frequency
                               # total frames = N_CHILDREN × MIN_SAMPLES_PER_CYCLE

# Scene geometry
ROOT_SCALE   = 3.0   # scale of each topology root node
CHILD_SCALE  = 0.5   # scale of each child node
ROOT_SPACING = 50.0  # X distance between topology roots

# Geometry IDs (ANTz kNPgeo* enum — see glyphviz/geometry.py)
GEO_CUBE     = 1
GEO_SPHERE   = 3
GEO_CONE     = 5
GEO_TORUS    = 7
GEO_PIN      = 16
GEO_CYLINDER = 19
GEO_POINT    = 22

# Topology roots: (topo_id, geo_id, name, (r,g,b), tx_lo, tx_hi, periodic)
#
# translate_x interpretation per topology:
#   Sphere / Point : longitude in degrees  (0–360)
#   Torus          : major-ring angle in degrees  (0–360)
#   Rod            : axial position  (0 = bottom, 180 = top)
#   Pin            : height above pin-head (scene units)
#   Plane          : Cartesian X offset (scene units)
#   Spiral         : helix angle in degrees  (720 = two full turns)
#
# periodic=True  → children fill [tx_lo, tx_hi) — closed loop, no duplicate endpoints
# periodic=False → children fill [tx_lo, tx_hi] — open line, endpoints included

TOPOLOGY_ROOTS = [
    # topo  geo          name       color             tx_lo   tx_hi  periodic
    (  2,  GEO_SPHERE,  'Sphere',  ( 50, 100, 220),   0.0,  360.0,  True  ),
    (  3,  GEO_TORUS,   'Torus',   (220, 120,  50),   0.0,  360.0,  True  ),
    (  5,  GEO_PIN,     'Pin',     ( 50, 200,  80),   1.0,   16.0,  False ),
    (  6,  GEO_CYLINDER,'Rod',     (220, 200,  50),   0.0,  180.0,  False ),
    (  7,  GEO_POINT,   'Point',   (200,  50, 200),   0.0,  360.0,  True  ),
    (  8,  GEO_CUBE,    'Plane',   ( 50, 200, 200),  -15.5,  15.5,  False ),
    ( 14,  GEO_CONE,    'Spiral',  (220,  80,  80),   0.0,  720.0,  True  ),
]

## Derived constants and helpers

In [ ]:
N_ROOTS    = len(TOPOLOGY_ROOTS)
N_TRACKS   = N_CHILDREN
NUM_FRAMES = N_TRACKS * MIN_SAMPLES_PER_CYCLE

print(f'Roots: {N_ROOTS}   Children per root: {N_CHILDREN}   '
      f'Total nodes: {N_ROOTS + N_ROOTS * N_CHILDREN}')
print(f'Tracks: {N_TRACKS}   Frames: {NUM_FRAMES}   '
      f'(track {N_TRACKS} Hz → {MIN_SAMPLES_PER_CYCLE} frames/cycle)')

def _root_x(ri):
    return (ri - (N_ROOTS - 1) / 2.0) * ROOT_SPACING

def _child_tx(tx_lo, tx_hi, periodic, ci):
    if periodic:
        return tx_lo + ci * (tx_hi - tx_lo) / N_CHILDREN
    return tx_lo + ci * (tx_hi - tx_lo) / max(N_CHILDREN - 1, 1)

## 1 · Node file (`gv_node.csv`)

Key fields:
- `ch_input_id` — links a child node to a channel.  Child *i* (1-indexed) gets `ch_input_id = i`, so all seven topology versions of child 1 share channel 1 (track 1, 1 Hz).
- `topo` — the parent topology that arranges its children in 3-D space.
- `geometry` — the glyph shape drawn for the node.

In [ ]:
def write_nodes(path):
    fieldnames = [
        'id', 'type', 'parent_id', 'branch_level',
        'translate_x', 'translate_y', 'translate_z',
        'rotate_x', 'rotate_y', 'rotate_z',
        'scale_x', 'scale_y', 'scale_z',
        'color_r', 'color_g', 'color_b', 'color_a',
        'geometry', 'hide', 'topo', 'ratio', 'ch_input_id',
    ]
    rows = []
    nid  = 1
    for ri, (topo, geo, name, color, tx_lo, tx_hi, periodic) in enumerate(TOPOLOGY_ROOTS):
        root_id = nid
        rows.append({
            'id': root_id, 'type': 5, 'parent_id': 0, 'branch_level': 0,
            'translate_x': round(_root_x(ri), 4),
            'translate_y': 0.0, 'translate_z': 0.0,
            'rotate_x': 0.0, 'rotate_y': 0.0, 'rotate_z': 0.0,
            'scale_x': ROOT_SCALE, 'scale_y': ROOT_SCALE, 'scale_z': ROOT_SCALE,
            'color_r': color[0], 'color_g': color[1], 'color_b': color[2], 'color_a': 255,
            'geometry': geo, 'hide': 0, 'topo': topo, 'ratio': 0.15, 'ch_input_id': 0,
        })
        nid += 1
        for ci in range(N_CHILDREN):
            rows.append({
                'id': nid, 'type': 5, 'parent_id': root_id, 'branch_level': 1,
                'translate_x': round(_child_tx(tx_lo, tx_hi, periodic, ci), 4),
                'translate_y': 0.0, 'translate_z': 0.0,
                'rotate_x': 0.0, 'rotate_y': 0.0, 'rotate_z': 0.0,
                'scale_x': CHILD_SCALE, 'scale_y': CHILD_SCALE, 'scale_z': CHILD_SCALE,
                'color_r': 200, 'color_g': 200, 'color_b': 200, 'color_a': 255,
                'geometry': GEO_SPHERE, 'hide': 0, 'topo': 0, 'ratio': 0.1,
                'ch_input_id': ci + 1,
            })
            nid += 1
    with open(path, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)
    print(f'Written: {path.name}  ({nid - 1} nodes)')

write_nodes(OUTPUT_DIR / f'{PREFIX}_gv_node.csv')

## 2 · Tag file (`gv_tag.csv`)

One tag per topology root, showing the topology name in the 3-D viewport.
`record_id` matches the root node's `id`.

In [ ]:
def write_tags(path):
    fieldnames = ['id', 'table_id', 'record_id', 'title']
    rows = [{
        'id': ri, 'table_id': 0, 'record_id': ri + 1,
        'title': entry[2],
    } for ri, entry in enumerate(TOPOLOGY_ROOTS)]
    with open(path, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)
    print(f'Written: {path.name}  ({len(rows)} tags)')

write_tags(OUTPUT_DIR / f'{PREFIX}_gv_tag.csv')

## 3 · Channel map (`gv_ch-map.csv`)

Maps each channel ID to a track and an animated attribute.

| channel_id | track_id | attribute   | meaning |
|-----------|---------|-------------|--------|
| 1         | 1       | translate_z | child with ch_input_id=1 gets track 1 (1 Hz) |
| 2         | 2       | translate_z | child with ch_input_id=2 gets track 2 (2 Hz) |
| …         | …       | translate_z | … |
| 32        | 32      | translate_z | child with ch_input_id=32 gets track 32 (32 Hz) |

In [ ]:
def write_ch_map(path):
    fieldnames = ['id', 'channel_id', 'track_id', 'attribute',
                  'track_table_id', 'ch_map_table_id', 'record_id']
    rows = [{
        'id': i, 'channel_id': i + 1, 'track_id': i + 1,
        'attribute': 'translate_z',
        'track_table_id': 0, 'ch_map_table_id': 0, 'record_id': 0,
    } for i in range(N_TRACKS)]
    with open(path, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)
    print(f'Written: {path.name}  ({len(rows)} channel mappings)')

write_ch_map(OUTPUT_DIR / f'{PREFIX}_gv_ch-map.csv')

## 4 · Channel tracks (`gv_ch-tracks.csv`)

### Frequency design

To ensure the fastest track (32 Hz) has at least `MIN_SAMPLES_PER_CYCLE` samples per cycle:

$$\Delta t = \frac{1}{N_{\text{tracks}} \times S_{\text{min}}} \quad \Rightarrow \quad
N_{\text{frames}} = N_{\text{tracks}} \times S_{\text{min}} = 32 \times 16 = 512$$

For track $i$ (1-indexed) at frame $j$ (0-indexed):

$$v_{i,j} = A \cdot \sin\!\left(\frac{2\pi \cdot i \cdot j}{N_{\text{frames}}}\right)$$

| Track | Frequency | Frames per cycle |
|-------|-----------|------------------|
| 1     | 1 Hz      | 512              |
| 8     | 8 Hz      | 64               |
| 16    | 16 Hz     | 32               |
| 32    | 32 Hz     | 16               |

In [ ]:
def write_ch_tracks(path):
    track_cols = [f'ch{i + 1}' for i in range(N_TRACKS)]
    fieldnames = ['cyclecount'] + track_cols
    denom = float(NUM_FRAMES)
    rows = []
    for j in range(NUM_FRAMES):
        row = {'cyclecount': j}
        for i in range(N_TRACKS):
            angle = 2.0 * math.pi * (i + 1) * j / denom
            row[track_cols[i]] = round(AMPLITUDE * math.sin(angle), 6)
        rows.append(row)
    with open(path, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)
    print(f'Written: {path.name}  ({NUM_FRAMES} frames × {N_TRACKS} tracks, ±{AMPLITUDE} units)')

write_ch_tracks(OUTPUT_DIR / f'{PREFIX}_gv_ch-tracks.csv')

## Preview — first few track values

In [ ]:
print(f'Frequencies: 1 Hz (track 1, period={NUM_FRAMES} frames) → '
      f'{N_TRACKS} Hz (track {N_TRACKS}, period={MIN_SAMPLES_PER_CYCLE} frames)')
print(f'Total animation: {NUM_FRAMES} frames')
print()
print('Frame  Track-1 (1Hz)  Track-8 (8Hz)  Track-16 (16Hz)  Track-32 (32Hz)')
print('-' * 68)
sample_frames = [0, 16, 32, 64, 128, 256]
for j in sample_frames:
    def val(i):
        return round(AMPLITUDE * math.sin(2 * math.pi * i * j / NUM_FRAMES), 3)
    print(f'{j:5d}  {val(1):+.3f}         {val(8):+.3f}         '
          f'{val(16):+.3f}           {val(32):+.3f}')

## Adapting this example

| Goal | What to change |
|------|---------------|
| More/fewer children | `N_CHILDREN` (also changes `N_TRACKS` and `NUM_FRAMES`) |
| Larger oscillations | `AMPLITUDE` |
| Smoother animation at high freq | `MIN_SAMPLES_PER_CYCLE` (increase, e.g. 32) |
| Animate a different attribute | `'attribute': 'translate_z'` in `write_ch_map()` → try `'scale_z'` or `'color_r'` |
| Different topology arrangement | Edit `tx_lo`, `tx_hi`, `periodic` in `TOPOLOGY_ROOTS` |
| Only one topology | Remove entries from `TOPOLOGY_ROOTS` |